# Multilingual trustworthiness study: results

Walks through the study results in pipeline order. The notebook only **reads** `results/`;
the numbered scripts produce it:

| Step | Script | What it does | Paid |
|---|---|---|---|
| 01 | `scripts/01_validate_inputs.py` | Validates corpus, annotations, config and validation sets | no |
| 02 | `scripts/02_generate.py` | Generates responses (Sol via API; Qwen, Gemma on GPU) | Sol only |
| 03 | `scripts/03_validate_judge.py` | Scores validation fixtures and measures judge agreement | `run` only |
| 04 | `scripts/04_score.py` | Scores every study response with the validated judge | yes |
| 05 | `scripts/05_status.py` | Integrity checks and `results/STATUS.md` | no |
| 06 | `scripts/06_analyze.py` | Prespecified analysis, tables and figures | no |
| 07 | `scripts/07_qa.py` | Reviewer QA sample compared with the judge | no |
| 08 | `scripts/08_robustness.py` | Post hoc robustness checks and manuscript figures | no |

Run the cell below first; it locates the repository and imports the steps.

In [ ]:
import importlib.util, json, sys
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/common.py').exists())
sys.path.insert(0, str(ROOT / 'scripts'))
RESULTS = ROOT / 'results'

def step(name):
    spec = importlib.util.spec_from_file_location('step_' + name, ROOT / 'scripts' / f'{name}.py')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

inputs, judge, status, analysis = (step(n) for n in ('01_validate_inputs', '03_validate_judge', '05_status', '06_analyze'))

## 1. Inputs

500 tasks in three matched language conditions; the study uses the 157 tasks with accepted gold annotations.

In [ ]:
assert inputs.main() == 0

## 2. Judge validation

The judge procedure (`scoring-0.5`) is accepted only if ordinal alpha ≥ 0.80, critical-RIU exact agreement ≥ 0.90
and safety-check exact agreement = 1.00. Three assessments are recorded: the original 30-fixture calibration
(earlier procedure, failed on safety checks), the 88-fixture held-out set against the first references, and the
held-out set after the blind re-rate of the disputed fixtures.

In [ ]:
rows = []
for args in (('calibration', 'scoring-0.3', False), ('heldout', 'scoring-0.5', True), ('heldout', 'scoring-0.5', False)):
    r = judge.assess(*args)
    rows.append(f"| {r['set']} | {r['procedure']} | {r['references']} | {r['ordinal_alpha']:.4f} | "
                f"{r['critical_exact']:.4f} | {r['safety_exact']:.4f} | {'yes' if r['passes'] else 'no'} |")
display(Markdown('| Set | Procedure | References | Alpha | Critical exact | Safety exact | Passes |\n'
                 '|---|---|---|---:|---:|---:|---|\n' + '\n'.join(rows)))

## 3. Status and integrity

Every primary cell must have exactly one response and one valid score made from that response.

In [ ]:
code_ = status.main()
display(Markdown((RESULTS / 'STATUS.md').read_text()))
assert code_ == 0

## 4. Prespecified analysis

Primary estimand: critical omission rate, BN − EN, paired by task, domain-equal, three primary models weighted
equally, with 95% task-clustered bootstrap CIs. Re-running takes a few seconds.

In [ ]:
report = analysis.analyze()
display(Markdown(analysis.summary(report, analysis.load_models())))

In [ ]:
out = RESULTS / 'analysis'
analysis.figures(report, analysis.load_models(), out)
display(Image(out / 'critical_omission_effects.png'), Image(out / 'critical_coverage_by_language.png'))

## 5. Reading the results

- The primary estimand compares *literal omission* of critical information. Content that is present but wrong or
  insufficient is counted separately, so compare the omission rows with the critical-coverage rows.
- Model-specific estimates are reported next to the panel estimate; a pooled effect is not read as universal when
  models differ.
- Gemini is exploratory (20 tasks) and is shown descriptively only.
- Scores are automated. A reviewer QA sample of 60 responses is compared with the judge in `results/qa/summary.md`,
  and the post hoc robustness checks are in `results/robustness/summary.md`.